In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3


In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()


OpenAI API Key exists and begins sk-proj-


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

### Authentication

In [4]:
AUTHORIZED_USERS = {
    "admin": "admin123",
    "manager": "manager456",
    "staff": "staff789"
}

def authenticate_user(username, password):
    """Check if username and password are valid"""
    return username in AUTHORIZED_USERS and AUTHORIZED_USERS[username] == password

In [5]:
DB = "prices.db"

### Tool


In [6]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [7]:
def set_ticket_price(city, price, username=None, password=None):
    # Check authentication if credentials are provided
    if username and password:
        if not authenticate_user(username, password):
            return "Authentication failed. You do not have the right to change ticket prices."
    else:
        return "Authentication required. Username and password are needed to change ticket prices."
    
    # If authentication successful, proceed with price update
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
    return f"Ticket price to {city} has been successfully set to ${price}"

In [ ]:
get_price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a flight ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a flight ticket to the destination city. Call this when you need to set the ticket price, for example when a customer asks 'Set the price of a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to set the ticket price to",
            },
            "price": {
                "type": "number",
                "description": "The price of the flight ticket to the destination city",
            },
            "username": {
                "type": "string",
                "description": "The username for authentication. Required to change ticket prices.",
            },
            "password": {
                "type": "string",
                "description": "The password for authentication. Required to change ticket prices.",
            },
        },
        "required": ["destination_city", "price", "username", "password"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": get_price_function}, {"type": "function", "function": set_price_function}]

In [9]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get('price')
            username = arguments.get('username')
            password = arguments.get('password')
            result = set_ticket_price(city, price, username, password)
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id
            })
    return responses

In [14]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print('User message:', message)
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    print('Response:', response.choices[0].message.content)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


User message: Hi
Response: Hello! How can I assist you today?
User message: I want to know the ticket price to London and Tokyo
DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Getting price for Tokyo
Response: The ticket price to London is $699, and to Tokyo is $1400.
User message: I want to change the ticket prices
Response: Please provide the new prices along with your username and password for authentication.
User message: change the LOndon ticket price to 700$ and Tokyo ticket price to 1500$
Response: Please provide your username and password to proceed with changing the ticket prices.
User message: username: admin and password: admin123
Response: The ticket price to London has been successfully set to $700, and the price to Tokyo is now $1500.
User message: Can you check the ticket price to Hyderabad and only if it is more than Tokyo's ticket price, check London ticket price
DATABASE TOOL CALLED: Getting price for Hyderabad
Response: The ticket price to Hydera